In [4]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Load data
df = pd.read_csv('../data/cleaned/main_dataset.csv')
df['body'] = df['body'].str.strip().str.title()

# Variables
season_order = ['Winter', 'Spring', 'Summer']  # Fall has no data, removed
colors = {'Winter':'#5B8DB8','Spring':'#6DBF8A','Summer':'#F5A623'}

# --- Data prep ---
season_counts = df['season'].value_counts().reindex(season_order).dropna()
avg_prices = df.groupby('season')['sellingprice'].mean().reindex(season_order).dropna()

top_bodies = df['body'].value_counts().head(6).index
df_top = df[df['body'].isin(top_bodies)]
pivot = df_top.groupby(['season','body']).size().reset_index(name='count')

monthly = df.groupby(['year_sold','month']).agg(
    sales_count=('sellingprice','count'),
    avg_gas=('gas_price','mean')
).reset_index()
monthly['date'] = pd.to_datetime(
    monthly[['year_sold','month']].assign(day=1).rename(columns={'year_sold':'year'})
)

# Filter gas price to match sales date range only
min_date = monthly['date'].min()
max_date = monthly['date'].max()
monthly = monthly[(monthly['date'] >= min_date) & (monthly['date'] <= max_date)]

body_counts = df['body'].value_counts().head(8)

total_sales = f"{len(df):,}"
avg_price = f"${df['sellingprice'].mean():,.0f}"
top_season = df['season'].value_counts().index[0]
top_body = df['body'].value_counts().index[0]

print("All variables ready!")
print("Seasons:", season_counts.to_dict())
print("Date range:", min_date, "→", max_date)

All variables ready!
Seasons: {'Winter': 348409, 'Spring': 98571, 'Summer': 98624}
Date range: 2014-01-01 00:00:00 → 2015-07-01 00:00:00


In [5]:
# --- Build charts ---
fig1 = go.Figure(go.Bar(
    x=season_counts.index, y=season_counts.values,
    marker_color=[colors[s] for s in season_order],
    text=[f'{v:,}' for v in season_counts.values],
    textposition='outside'
))
fig1.update_layout(title=None, height=350, margin=dict(t=20,b=40,l=40,r=20),
                   showlegend=False, plot_bgcolor='white', paper_bgcolor='white')
fig1.update_yaxes(showgrid=True, gridcolor='#f0f0f0')

fig2 = go.Figure(go.Bar(
    x=avg_prices.index, y=avg_prices.values,
    marker_color=[colors[s] for s in season_order],
    text=[f'${v:,.0f}' for v in avg_prices.values],
    textposition='outside'
))
fig2.update_layout(title=None, height=350, margin=dict(t=20,b=40,l=40,r=20),
                   showlegend=False, plot_bgcolor='white', paper_bgcolor='white')
fig2.update_yaxes(showgrid=True, gridcolor='#f0f0f0')

fig3 = make_subplots(specs=[[{"secondary_y": True}]])
fig3.add_trace(go.Scatter(x=monthly['date'], y=monthly['sales_count'],
    name='Sales Volume', line=dict(color='#5B8DB8', width=2)), secondary_y=False)
fig3.add_trace(go.Scatter(x=monthly['date'], y=monthly['avg_gas'],
    name='Gas Price', line=dict(color='#E07B5A', width=2, dash='dash')), secondary_y=True)
fig3.update_layout(title=None, height=350, margin=dict(t=20,b=40,l=40,r=60),
                   plot_bgcolor='white', paper_bgcolor='white',
                   legend=dict(orientation='h', y=1.1))
fig3.update_yaxes(title_text='Sales Volume', secondary_y=False, gridcolor='#f0f0f0')
fig3.update_yaxes(title_text='Gas Price (USD)', secondary_y=True)

fig4 = go.Figure(go.Pie(
    values=body_counts.values, labels=body_counts.index,
    hole=0.45, textposition='outside'
))
fig4.update_layout(title=None, height=350, margin=dict(t=20,b=20,l=20,r=20),
                   paper_bgcolor='white')

# --- Build HTML ---
html = f"""
<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8">
  <title>Automobile Seasonal Demand Dashboard</title>
  <style>
    * {{ margin: 0; padding: 0; box-sizing: border-box; }}
    body {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
            background: #f4f6f9; color: #333; }}
    .header {{ background: linear-gradient(135deg, #1a237e, #283593);
               color: white; padding: 32px 40px; }}
    .header h1 {{ font-size: 28px; font-weight: 700; margin-bottom: 6px; }}
    .header p  {{ font-size: 14px; opacity: 0.8; }}
    .kpi-row {{ display: flex; gap: 20px; padding: 24px 40px; }}
    .kpi {{ background: white; border-radius: 12px; padding: 20px 28px;
            flex: 1; box-shadow: 0 2px 8px rgba(0,0,0,0.07); }}
    .kpi .label {{ font-size: 12px; color: #888; text-transform: uppercase;
                   letter-spacing: 0.5px; margin-bottom: 8px; }}
    .kpi .value {{ font-size: 28px; font-weight: 700; color: #1a237e; }}
    .kpi .sub   {{ font-size: 12px; color: #aaa; margin-top: 4px; }}
    .charts-grid {{ display: grid; grid-template-columns: 1fr 1fr;
                    gap: 20px; padding: 0 40px 40px; }}
    .chart-card {{ background: white; border-radius: 12px; padding: 8px;
                   box-shadow: 0 2px 8px rgba(0,0,0,0.07); }}
    .chart-card.wide {{ grid-column: span 2; }}
    h2 {{ font-size: 14px; font-weight: 600; color: #555;
          padding: 12px 16px 0; text-transform: uppercase; letter-spacing: 0.5px; }}
  </style>
</head>
<body>
<div class="header">
  <h1>Automobile Seasonal Demand Analysis</h1>
  <p>Interactive dashboard — Sales, Vehicle Types, Fuel Prices and Seasonal Trends</p>
</div>
<div class="kpi-row">
  <div class="kpi">
    <div class="label">Total Vehicles Sold</div>
    <div class="value">{total_sales}</div>
    <div class="sub">Across all seasons</div>
  </div>
  <div class="kpi">
    <div class="label">Average Selling Price</div>
    <div class="value">{avg_price}</div>
    <div class="sub">All vehicle types</div>
  </div>
  <div class="kpi">
    <div class="label">Peak Season</div>
    <div class="value">{top_season}</div>
    <div class="sub">Highest sales volume</div>
  </div>
  <div class="kpi">
    <div class="label">Top Vehicle Type</div>
    <div class="value">{top_body}</div>
    <div class="sub">Most sold body type</div>
  </div>
</div>
<div class="charts-grid">
  <div class="chart-card">
    <h2>Sales Volume by Season</h2>
    {fig1.to_html(full_html=False, include_plotlyjs='cdn')}
  </div>
  <div class="chart-card">
    <h2>Average Price by Season</h2>
    {fig2.to_html(full_html=False, include_plotlyjs=False)}
  </div>
  <div class="chart-card wide">
    <h2>Gas Price vs Sales Volume Over Time</h2>
    {fig3.to_html(full_html=False, include_plotlyjs=False)}
  </div>
  <div class="chart-card">
    <h2>Vehicle Type Breakdown</h2>
    {fig4.to_html(full_html=False, include_plotlyjs=False)}
  </div>
</div>
</body>
</html>
"""

with open('../dashboards/automobile_dashboard.html', 'w') as f:
    f.write(html)

print("Dashboard saved! ✓")

Dashboard saved! ✓


In [6]:
import subprocess
subprocess.run(['open', '../dashboards/automobile_dashboard.html'])

CompletedProcess(args=['open', '../dashboards/automobile_dashboard.html'], returncode=0)